# Debugging Neural Networks

## Objectives
- Detect and fix NaN losses
- Check gradients for vanishing/exploding
- Visualize activations
- Analyze loss curves
- Use logging and monitoring

## Introduction
Debugging is critical when training neural networks. This notebook covers common issues and diagnostic techniques.

## What We're About to Do

The code below imports essential libraries. These libraries provide the foundational tools for tensor operations and neural network construction. Pay attention to what each import provides – understanding dependencies helps you know what's available for solving problems.


In [ ]:
# Import necessary libraries for tensor operations and deep learning
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")


In [ ]:
# Visualize the results
plt.figure(figsize=(12, 4))
plt.plot(range(len(result)), result, marker='o', linestyle='-', linewidth=2)
plt.title('Results Visualization', fontsize=14, fontweight='bold')
plt.xlabel('Index')
plt.ylabel('Value')
plt.grid(True, alpha=0.3)
plt.show()

print('Visualization shows the pattern and distribution of results.')


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 1. Problem: NaN Loss (Exploding Gradients)

class UnstableModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 100)
        self.fc2 = nn.Linear(100, 100)
        self.fc3 = nn.Linear(100, 2)
        
        # Initialize with very large weights
# Iterate through batches of data
        for param in self.parameters():
            nn.init.uniform_(param, -10, 10)
    
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

model_unstable = UnstableModel()
print("Unstable model created (large weight initialization)")

# Test forward pass
x = torch.randn(2, 10)
out = model_unstable(x)
print(f"Output: {out}")
print(f"Output has NaN: {torch.isnan(out).any()}")
print(f"Output has Inf: {torch.isinf(out).any()}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 2. Solution: Proper Initialization

class StableModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 100)
        self.fc2 = nn.Linear(100, 100)
        self.fc3 = nn.Linear(100, 2)
        
        # Proper initialization
# Iterate through batches of data
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)
    
    def forward(self, x):
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc3(x)
        return x

model_stable = StableModel()
print("Stable model created (proper initialization)")

# Test forward pass
out = model_stable(x)
print(f"Output: {out}")
print(f"Output has NaN: {torch.isnan(out).any()}")
print(f"Output has Inf: {torch.isinf(out).any()}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 3. Gradient Analysis

def analyze_gradients(model, criterion, x, y):
    """Analyze gradient statistics"""
    output = model(x)
# Compute the loss (error) between predictions and actual values
    loss = criterion(output, y)
    loss.backward()
    
    gradient_stats = {}
# Iterate through batches of data
    for name, param in model.named_parameters():
        if param.grad is not None:
# Update model parameters based on computed gradients
            grad = param.grad.data
            gradient_stats[name] = {
                'mean': grad.mean().item(),
                'std': grad.std().item(),
                'min': grad.min().item(),
                'max': grad.max().item(),
                'norm': grad.norm().item()
            }
    
    return gradient_stats

# Test on stable model
x = torch.randn(32, 10)
y = torch.randint(0, 2, (32,))
# Compute the loss (error) between predictions and actual values
criterion = nn.CrossEntropyLoss()

model_stable.zero_grad()
gradient_stats = analyze_gradients(model_stable, criterion, x, y)

print("\nGradient Statistics (Stable Model):")
# Iterate through batches of data
for name, stats in gradient_stats.items():
    print(f"\n{name}:")
# Iterate through batches of data
    for key, val in stats.items():
        print(f"  {key}: {val:.6f}")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 4. Gradient Clipping

def train_with_gradient_clipping(epochs=10, clip_norm=1.0):
    """Train model with gradient clipping"""
    model = StableModel()
# Update model parameters based on computed gradients
    optimizer = optim.Adam(model.parameters(), lr=0.001)
# Compute the loss (error) between predictions and actual values
    criterion = nn.CrossEntropyLoss()
    
    # Create dataset
    X = torch.randn(200, 10)
    y = torch.randint(0, 2, (200,))
    loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
    
# Compute the loss (error) between predictions and actual values
    losses = []
    grad_norms = []
    
# Iterate through batches of data
    for epoch in range(epochs):
# Compute the loss (error) between predictions and actual values
        epoch_loss = 0
        epoch_grad_norm = 0
        
# Iterate through batches of data
        for X_batch, y_batch in loader:
            output = model(X_batch)
# Compute the loss (error) between predictions and actual values
            loss = criterion(output, y_batch)
            
            optimizer.zero_grad()
            loss.backward()
            
            # Gradient clipping
# Update model parameters based on computed gradients
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), clip_norm)
            optimizer.step()
            
# Compute the loss (error) between predictions and actual values
            epoch_loss += loss.item()
            epoch_grad_norm += grad_norm.item()
        
# Compute the loss (error) between predictions and actual values
        avg_loss = epoch_loss / len(loader)
        avg_grad = epoch_grad_norm / len(loader)
        losses.append(avg_loss)
        grad_norms.append(avg_grad)
        
        if (epoch + 1) % 5 == 0:
# Compute the loss (error) between predictions and actual values
            print(f"Epoch {epoch+1}: Loss={avg_loss:.4f}, Grad Norm={avg_grad:.4f}")
    
    return losses, grad_norms

# Compute the loss (error) between predictions and actual values
losses_clipped, grad_norms = train_with_gradient_clipping(epochs=10, clip_norm=1.0)
print("Training completed with gradient clipping.")


## The Training Process

This is the core learning loop. We'll forward-pass data through the model, compute loss, backpropagate gradients, and update parameters. This iterative process gradually improves the model.


In [ ]:
# Execute the training loop with proper tracking
## 5. Visualize Gradient Flow

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(grad_norms, linewidth=2.5, marker='o', markersize=6)
ax.set_xlabel('Epoch')
ax.set_ylabel('Gradient Norm')
ax.set_title('Gradient Norm During Training (With Clipping)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 6. Activation Visualization

class HookModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(10, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 2)
        
        # Initialize properly
# Iterate through batches of data
        for layer in [self.fc1, self.fc2, self.fc3]:
            nn.init.xavier_uniform_(layer.weight)
        
        self.activations = {}
    
    def forward(self, x):
        x = self.fc1(x)
        self.activations['fc1'] = x.clone().detach()
        x = F.relu(x)
        
        x = self.fc2(x)
        self.activations['fc2'] = x.clone().detach()
        x = F.relu(x)
        
        x = self.fc3(x)
        self.activations['fc3'] = x.clone().detach()
        
        return x

model_hook = HookModel()
x_test = torch.randn(128, 10)
output = model_hook(x_test)

# Analyze activations
print("\nActivation Statistics:")
# Iterate through batches of data
for layer_name, activation in model_hook.activations.items():
    print(f"\n{layer_name}:")
    print(f"  Shape: {activation.shape}")
    print(f"  Mean: {activation.mean().item():.4f}")
    print(f"  Std: {activation.std().item():.4f}")
    print(f"  Min: {activation.min().item():.4f}")
    print(f"  Max: {activation.max().item():.4f}")
    print(f"  % zeros: {(activation == 0).float().mean().item() * 100:.2f}%")


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
## 7. Visualize Activation Distributions

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Iterate through batches of data
for idx, (layer_name, activation) in enumerate(model_hook.activations.items()):
    axes[idx].hist(activation.view(-1).numpy(), bins=50, edgecolor='black', alpha=0.7)
    axes[idx].set_title(f'{layer_name} Activation Distribution')
    axes[idx].set_xlabel('Activation Value')
    axes[idx].set_ylabel('Frequency')
    axes[idx].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Define a custom function with detailed implementation
## 8. Detecting Dead ReLU

def detect_dead_neurons(activations):
    """Detect dead ReLU neurons"""
    dead_ratios = {}
# Iterate through batches of data
    for layer_name, activation in activations.items():
        # After ReLU (non-negative values)
        if 'relu' in layer_name or layer_name.endswith('out'):
            dead_ratio = (activation == 0).float().mean().item()
            dead_ratios[layer_name] = dead_ratio
    return dead_ratios

print("\nDead Neuron Detection:")
# Iterate through batches of data
for layer_name, activation in model_hook.activations.items():
    # Check if this is after ReLU
    zero_count = (activation == 0).sum().item()
    total = activation.numel()
    zero_ratio = zero_count / total
    print(f"{layer_name}: {zero_ratio*100:.2f}% zeros (potential dead neurons)")

print("\nNote: Some zeros are expected, but > 50% indicates dead ReLU problem.")


## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


In [ ]:
# Configure loss function and optimization algorithm
## 9. Loss Curve Analysis

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Normal training
# Compute the loss (error) between predictions and actual values
epochs = np.arange(len(losses_clipped))
# Compute the loss (error) between predictions and actual values
axes[0, 0].plot(epochs, losses_clipped, linewidth=2, marker='o')
axes[0, 0].set_title('Normal Training (Decreasing Loss)')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss')
axes[0, 0].grid(True, alpha=0.3)

# Exploding loss
# Compute the loss (error) between predictions and actual values
bad_losses = [1.0, 1.1, 1.3, 1.8, 3.2, 10.5, np.nan, np.nan]
# Compute the loss (error) between predictions and actual values
axes[0, 1].plot(bad_losses, linewidth=2, marker='o', color='red')
axes[0, 1].set_title('Exploding Loss (Training Unstable)')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].grid(True, alpha=0.3)

# Plateauing loss
# Compute the loss (error) between predictions and actual values
# Iterate through batches of data
plateau_losses = [1.0] + [0.9 - 0.08*i for i in range(8)] + [0.2]*10
# Compute the loss (error) between predictions and actual values
axes[1, 0].plot(plateau_losses, linewidth=2, marker='o', color='orange')
axes[1, 0].set_title('Plateauing Loss (May Need LR Schedule)')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].grid(True, alpha=0.3)

# Oscillating loss
# Iterate through batches of data
oscillating = [1.0 - 0.05*i + 0.3*np.sin(i) for i in range(20)]
axes[1, 1].plot(oscillating, linewidth=2, marker='o', color='green')
axes[1, 1].set_title('Oscillating Loss (Learning Rate Too High)')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Building the Model

Now we'll define our neural network architecture. Each layer transforms the input in a specific way, building up complexity. The order and configuration of layers directly determines what patterns the model can learn.


In [ ]:
# Set up the neural network model architecture
## 10. Debugging Checklist

debugging_checklist = """
╔═══════════════════════════════════════════════════════════════════════════╗
║                      DEBUGGING CHECKLIST                                  ║
╚═══════════════════════════════════════════════════════════════════════════╝

1. DATA ISSUES:
   □ Check data shape and types
# Iterate through batches of data
   □ Look for NaN/Inf values in data
   □ Verify label distribution (class imbalance?)
   □ Check normalization/scaling

2. MODEL INITIALIZATION:
   □ Use Xavier/Kaiming initialization
   □ Initialize biases to zero
   □ Avoid very large or very small weights
# Iterate through batches of data
   □ Use BatchNorm for stable training

3. TRAINING STABILITY:
# Iterate through batches of data
   □ Check for NaN/Inf in loss
   □ Monitor gradient norms
   □ Detect exploding/vanishing gradients
   □ Use gradient clipping if needed
   □ Try lower learning rate

4. LOSS CURVE:
   □ Should generally decrease over time
# Iterate through batches of data
   □ Watch for sudden spikes or NaN
   □ Check for oscillation (high LR)
   □ Look for plateau (need LR schedule)

5. VALIDATION:
   □ Compare train vs validation performance
   □ Use early stopping
   □ Monitor metrics beyond loss
# Iterate through batches of data
   □ Check for overfitting

6. COMMON FIXES:
   - NaN loss → Lower LR, check initialization, clip gradients
   - Oscillating loss → Reduce LR
   - Plateauing → Use LR schedule, try different LR
   - Slow convergence → Increase LR, use better optimizer
   - Dead ReLUs → Use LeakyReLU, better initialization
   - Overfitting → Add regularization, more data, smaller model
"""

print(debugging_checklist)


## Defining the Loss Function

The loss function measures how wrong our predictions are. During training, we'll minimize this value. Different tasks need different loss functions – the one we choose defines what 'good performance' means for our model.


## 🎯 Key Takeaways

✅ **Understanding fundamentals is crucial** – The concepts covered here form the foundation for all advanced deep learning techniques.

✅ **Each component has a specific purpose** – Whether it's data loading, model architecture, or optimization, every piece serves a distinct function in the pipeline.

✅ **Experimentation drives learning** – Don't just read the code; modify it, break it, and see what happens. That's how intuition develops.

✅ **Deep learning is iterative** – Success comes from systematically trying approaches, measuring results, and refining based on evidence.

✅ **Connect concepts, don't memorize** – Understanding how PyTorch tensors relate to autograd, which relates to neural networks, which connects to training loops, is far more valuable than memorizing individual APIs.

✅ **Performance matters in practice** – Once you understand the theory, optimizing for speed, memory, and scalability becomes crucial for real-world applications.


In [ ]:
# Configure loss function and optimization algorithm
## Key Takeaways
- Proper initialization prevents training instability
- Gradient clipping helps with exploding gradients
# Iterate through batches of data
- Monitor loss curves for training anomalies
- Visualize activations to detect dead neurons
- Use systematic debugging approach

## Interview Q&A

**Q1: What causes NaN loss during training?**
NaN usually comes from exploding gradients or invalid operations. Fix by: using proper initialization (Xavier/Kaiming), reducing learning rate, adding gradient clipping, or using batch normalization.

**Q2: How do you detect vanishing gradients?**
Monitor gradient magnitudes during training. If gradients become very small (< 1e-7), early layers stop learning. Use gradient checkpoints, batch norm, or skip connections.

**Q3: What does a healthy loss curve look like?**
Loss should smoothly decrease, eventually plateauing. Watch for: sudden spikes (unstable), oscillations (LR too high), no change (LR too low), or NaN (training failed).

## References
- [PyTorch Debugging Guide](https://pytorch.org/docs/stable/nn.init.html)
- [Understanding Gradient Flow](https://arxiv.org/abs/1502.01852)
